# 🧮 Math Escalation — TRL GRPO + Unsloth Training

This notebook demonstrates how to train an LLM on the **Math Escalation** environment using Reinforcement Learning. We follow the Meta OpenEnv Hackathon guidelines for a "perfect" project.

### 🌟 Features:
- **Environment-First**: Connects to the OpenEnv FastAPI server.
- **GRPO Training**: Uses the Group Relative Policy Optimization (GRPO) algorithm from TRL.
- **Unsloth Efficiency**: Leverages Unsloth for 2x faster training and 70% memory savings.
- **Multi-Component Reward**: Rewards both correctness and reasoning quality.
- **Session Isolation**: Uses unique episode IDs per rollout to support parallel GRPO without interference.

In [ ]:
# Cell 1 — Install Premium Dependencies
!pip install "unsloth[colab-new]" trl>=0.12.0 requests datasets matplotlib

In [ ]:
# Cell 2 — Configuration
# Update this if your server is running on a different URL (e.g. HF Space or Ngrok)
BASE_URL = "http://localhost:8000"

In [ ]:
# Cell 3 — Load Model with Unsloth
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# Cell 4 — Advanced Multi-Component Reward Function (Guideline #7, #9, #12)
import requests, re, math, uuid

def call_env(tool, args={}, episode_id=None):
    try:
        payload = {
            "action": {"type": "call_tool", "tool_name": tool, "arguments": args}
        }
        if episode_id:
            payload["episode_id"] = episode_id
            
        r = requests.post(f"{BASE_URL}/step", json=payload, timeout=5)
        return r.json()
    except Exception as e:
        return {"reward": -0.5, "observation": "Error"}

def compute_reward(completions, prompts, **kwargs):
    """
    Rewards correctness, formatting, and reasoning density.
    Each rollout uses a unique episode_id to prevent session interference.
    """
    rewards = []
    for i, (completion, prompt) in enumerate(zip(completions, prompts)):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        
        # Component 1: Reasoning Quality
        thought_reward = 0.0
        if "<thought>" in text and "</thought>" in text:
            thought_content = re.search(r"<thought>(.*?)</thought>", text, re.DOTALL)
            if thought_content and len(thought_content.group(1).strip()) > 50:
                thought_reward = 0.1
        else:
            thought_reward = -0.1
            
        # Component 2: Correctness
        clean_text = re.sub(r"<thought>.*?</thought>", "", text, flags=re.DOTALL)
        nums = re.findall(r"-?\d+\.?\d*", clean_text)
        
        if not nums:
            rewards.append(-0.5 + thought_reward)
            continue
            
        try:
            # Extract problem from prompt to sync server state
            prob_match = re.search(r"Problem: (.*?)\n", prompt)
            problem_str = prob_match.group(1) if prob_match else ""
            
            # Unique session for this specific rollout
            eid = f"rollout_{uuid.uuid4()}"
            requests.post(f"{BASE_URL}/reset", json={"episode_id": eid, "problem": problem_str})
            
            answer = float(nums[-1])
            resp = call_env("submit_answer", {"answer": answer}, episode_id=eid)
            env_reward = resp.get("reward", -0.5)
            
            rewards.append(float(env_reward) + thought_reward)
        except Exception:
            rewards.append(-0.5 + thought_reward)
    
    return rewards

In [ ]:
# Cell 5 — Build Dataset from Environment Curriculum
import datasets

def agent_solve_oracle(problem_str):
    """Local oracle to help build a curriculum-aware dataset."""
    clean = re.sub(r"[^0-9+\-*/().² ]", "", problem_str).strip()
    try:
        return float(eval(clean.replace("sqrt", "math.sqrt")))
    except: return 0.0

def build_dataset(n_problems=500):
    print(f"📊 Sampling {n_problems} problems from Environment...")
    data = []
    eid = "dataset_builder"
    requests.post(f"{BASE_URL}/reset", json={"episode_id": eid})
    
    for i in range(n_problems):
        resp = call_env("get_problem", episode_id=eid)
        obs = resp.get("observation", {})
        problem_str = obs.get("result", {}).get("data", "") if isinstance(obs, dict) else str(obs)
            
        prompt = (
            f"Solve this math problem. Provide your reasoning in <thought> tags, "
            f"and output the final numeric answer at the very end.\n\n"
            f"Problem: {problem_str}\n"
            f"Answer:"
        )
        data.append({"prompt": prompt})
        
        # Solve correctly 50% of the time to level up and get higher tier problems
        # If we always fail, we stay at Tier 1. If we always succeed, we rush to Tier 10.
        # We want a mix.
        if i % 2 == 0:
            # Use server state to get expected answer
            s_resp = call_env("get_status", episode_id=eid)
            # To get the answer from server without an 'oracle' tool, 
            # we can use the server's internal logic by just guessing or using a local oracle
            # Since we are building the dataset, let's just use a trick: 
            # The server logic is known. For simple arithmetic, eval is fine.
            # For complex tiers, we might need a better oracle.
            # But let's just solve it well enough to advance.
            ans = agent_solve_oracle(problem_str)
            call_env("submit_answer", {"answer": ans}, episode_id=eid)
        else:
            call_env("submit_answer", {"answer": -9999.0}, episode_id=eid)
        
        if (i+1) % 100 == 0:
            print(f"  Collected {i+1}/{n_problems}")
    
    return datasets.Dataset.from_list(data)

train_dataset = build_dataset(400)
print(f"Dataset Size: {len(train_dataset)}")

In [ ]:
# Cell 6 — GRPO Configuration & Training
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir = "./math_escalation_grpo",
    num_train_epochs = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    learning_rate = 5e-6,
    logging_steps = 1,
    save_steps = 50,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    report_to = "none",
    num_generations = 4,
    max_completion_length = 256,
    temperature = 0.9,
    gradient_checkpointing = True,
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    reward_funcs = compute_reward,
    processing_class = tokenizer,
)

print("🔥 Starting GRPO Training...")
trainer.train()
print("✅ Training Complete!")

In [ ]:
# Cell 7 — Save Merged Model
model.save_pretrained_merged("./math_grpo_merged", tokenizer, save_method = "merged_16bit")
print("💾 Model saved correctly at ./math_grpo_merged")